In [2]:
import tkinter as tk
from tkinter import ttk, messagebox
import time
import threading

# ─── Romania Map Data ───────────────────────────────────────────────────────

GRAPH = {
    'Arad':         {'Zerind': 75, 'Sibiu': 140, 'Timisoara': 118},
    'Zerind':       {'Arad': 75, 'Oradea': 71},
    'Oradea':       {'Zerind': 71, 'Sibiu': 151},
    'Sibiu':        {'Arad': 140, 'Oradea': 151, 'Fagaras': 99, 'Rimnicu Vilcea': 80},
    'Timisoara':    {'Arad': 118, 'Lugoj': 111},
    'Lugoj':        {'Timisoara': 111, 'Mehadia': 70},
    'Mehadia':      {'Lugoj': 70, 'Drobeta': 75},
    'Drobeta':      {'Mehadia': 75, 'Craiova': 120},
    'Craiova':      {'Drobeta': 120, 'Rimnicu Vilcea': 146, 'Pitesti': 138},
    'Rimnicu Vilcea': {'Sibiu': 80, 'Craiova': 146, 'Pitesti': 97},
    'Fagaras':      {'Sibiu': 99, 'Bucharest': 211},
    'Pitesti':      {'Rimnicu Vilcea': 97, 'Craiova': 138, 'Bucharest': 101},
    'Bucharest':    {'Fagaras': 211, 'Pitesti': 101, 'Giurgiu': 90, 'Urziceni': 85},
    'Giurgiu':      {'Bucharest': 90},
    'Urziceni':     {'Bucharest': 85, 'Hirsova': 98, 'Vaslui': 142},
    'Hirsova':      {'Urziceni': 98, 'Eforie': 86},
    'Eforie':       {'Hirsova': 86},
    'Vaslui':       {'Urziceni': 142, 'Iasi': 92},
    'Iasi':         {'Vaslui': 92, 'Neamt': 87},
    'Neamt':        {'Iasi': 87},
}

# Node positions scaled for a 900×600 canvas
NODE_POS = {
    'Arad':           (100, 160),
    'Zerind':         (160, 90),
    'Oradea':         (220, 40),
    'Sibiu':          (320, 220),
    'Timisoara':      (100, 300),
    'Lugoj':          (190, 370),
    'Mehadia':        (230, 440),
    'Drobeta':        (200, 510),
    'Craiova':        (310, 540),
    'Rimnicu Vilcea': (380, 340),
    'Fagaras':        (470, 220),
    'Pitesti':        (490, 390),
    'Bucharest':      (590, 420),
    'Giurgiu':        (560, 510),
    'Urziceni':       (700, 390),
    'Hirsova':        (790, 350),
    'Eforie':         (840, 440),
    'Vaslui':         (790, 260),
    'Iasi':           (790, 170),
    'Neamt':          (700, 110),
}

CITIES = sorted(GRAPH.keys())

# ─── Colors ─────────────────────────────────────────────────────────────────

BG          = "#0f1117"
PANEL_BG    = "#1a1d27"
ACCENT      = "#e8b84b"
NODE_DEF    = "#2a2d3e"
NODE_BORDER = "#404465"
TEXT_LIGHT  = "#e8eaf0"
TEXT_DIM    = "#6b6f8a"
EDGE_DEF    = "#2e3148"

COLOR_VISIT  = "#f59e0b"   # yellow  – currently visiting
COLOR_PATH   = "#8b5cf6"   # purple  – on current DLS path
COLOR_DEAD   = "#374151"   # dark    – dead-end / backtracked
COLOR_FINAL  = "#3b82f6"   # blue    – final solution path
COLOR_SOURCE = "#10b981"   # green   – source node
COLOR_DEST   = "#ef4444"   # red     – destination node


# ─── IDDFS Logic ────────────────────────────────────────────────────────────

def iddfs(graph, source, destination):
    """
    Yields animation frames as (event, data) tuples:
      ('new_limit', depth_limit)
      ('visit', node, path_so_far)
      ('backtrack', node, path_so_far)
      ('found', path)
      ('not_found',)
    """
    depth = 0
    while True:
        yield ('new_limit', depth)
        result = yield from _dls(graph, source, destination, depth, [source])
        if result == 'found':
            return
        if result == 'cutoff_not_possible':
            yield ('not_found',)
            return
        depth += 1


def _dls(graph, node, destination, limit, path):
    if node == destination:
        yield ('found', list(path))
        return 'found'
    if limit == 0:
        return 'cutoff'
    any_cutoff = False
    yield ('visit', node, list(path))
    neighbours = sorted(graph[node].keys())
    for child in neighbours:
        if child not in path:
            path.append(child)
            result = yield from _dls(graph, child, destination, limit - 1, path)
            path.pop()
            if result == 'found':
                return 'found'
            if result == 'cutoff':
                any_cutoff = True
            yield ('backtrack', node, list(path))
    return 'cutoff' if any_cutoff else 'cutoff_not_possible'


# ─── GUI Application ─────────────────────────────────────────────────────────

class RomaniaIDDFS(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("Romania Map – IDDFS Visualizer")
        self.configure(bg=BG)
        self.resizable(False, False)

        self._node_ovals  = {}
        self._node_labels = {}
        self._edge_lines  = {}
        self._node_colors = {}
        self._anim_speed  = 600   # ms between frames
        self._running     = False
        self._thread      = None

        self._build_ui()
        self._draw_map()

    # ── UI Layout ────────────────────────────────────────────────────────────

    def _build_ui(self):
        # Top bar
        top = tk.Frame(self, bg=PANEL_BG, pady=10)
        top.pack(fill='x')

        tk.Label(top, text="🗺  Romania IDDFS Visualizer",
                 font=("Georgia", 18, "bold"), bg=PANEL_BG, fg=ACCENT).pack(side='left', padx=20)

        ctrl = tk.Frame(top, bg=PANEL_BG)
        ctrl.pack(side='right', padx=20)

        tk.Label(ctrl, text="Source:", bg=PANEL_BG, fg=TEXT_LIGHT,
                 font=("Courier", 11)).grid(row=0, column=0, padx=6)
        self.src_var = tk.StringVar(value='Arad')
        ttk.Combobox(ctrl, textvariable=self.src_var, values=CITIES,
                     state='readonly', width=16).grid(row=0, column=1, padx=6)

        tk.Label(ctrl, text="Destination:", bg=PANEL_BG, fg=TEXT_LIGHT,
                 font=("Courier", 11)).grid(row=0, column=2, padx=6)
        self.dst_var = tk.StringVar(value='Bucharest')
        ttk.Combobox(ctrl, textvariable=self.dst_var, values=CITIES,
                     state='readonly', width=16).grid(row=0, column=3, padx=6)

        tk.Label(ctrl, text="Speed (ms):", bg=PANEL_BG, fg=TEXT_LIGHT,
                 font=("Courier", 11)).grid(row=0, column=4, padx=6)
        self.speed_var = tk.IntVar(value=600)
        tk.Scale(ctrl, from_=100, to=1500, orient='horizontal',
                 variable=self.speed_var, bg=PANEL_BG, fg=TEXT_LIGHT,
                 troughcolor=NODE_DEF, highlightthickness=0,
                 length=120).grid(row=0, column=5, padx=6)

        self.submit_btn = tk.Button(ctrl, text="  SUBMIT  ",
                                    font=("Courier", 11, "bold"),
                                    bg=ACCENT, fg=BG, relief='flat',
                                    padx=10, pady=4,
                                    command=self._on_submit)
        self.submit_btn.grid(row=0, column=6, padx=10)

        self.reset_btn = tk.Button(ctrl, text="  RESET  ",
                                   font=("Courier", 11, "bold"),
                                   bg=NODE_DEF, fg=TEXT_LIGHT, relief='flat',
                                   padx=10, pady=4,
                                   command=self._on_reset)
        self.reset_btn.grid(row=0, column=7, padx=4)

        # Canvas
        self.canvas = tk.Canvas(self, width=960, height=600,
                                bg=BG, highlightthickness=0)
        self.canvas.pack(padx=10, pady=(4, 0))

        # Status bar
        status_frame = tk.Frame(self, bg=PANEL_BG, pady=6)
        status_frame.pack(fill='x')

        self.status_var = tk.StringVar(value="Select source & destination, then press SUBMIT.")
        tk.Label(status_frame, textvariable=self.status_var,
                 bg=PANEL_BG, fg=TEXT_LIGHT,
                 font=("Courier", 11)).pack(side='left', padx=16)

        # Legend
        legend = tk.Frame(status_frame, bg=PANEL_BG)
        legend.pack(side='right', padx=16)
        items = [
            (COLOR_SOURCE, "Source"),
            (COLOR_DEST,   "Destination"),
            (COLOR_VISIT,  "Visiting"),
            (COLOR_PATH,   "Path"),
            (COLOR_DEAD,   "Dead-end"),
            (COLOR_FINAL,  "Solution"),
        ]
        for color, label in items:
            dot = tk.Canvas(legend, width=14, height=14, bg=PANEL_BG,
                            highlightthickness=0)
            dot.create_oval(2, 2, 13, 13, fill=color, outline='')
            dot.pack(side='left', padx=(6, 2))
            tk.Label(legend, text=label, bg=PANEL_BG, fg=TEXT_DIM,
                     font=("Courier", 9)).pack(side='left', padx=(0, 8))

    # ── Map Drawing ──────────────────────────────────────────────────────────

    def _draw_map(self):
        c = self.canvas
        r = 18   # node radius

        # Draw edges first
        drawn = set()
        for city, neighbours in GRAPH.items():
            for nb, dist in neighbours.items():
                key = tuple(sorted([city, nb]))
                if key in drawn:
                    continue
                drawn.add(key)
                x1, y1 = NODE_POS[city]
                x2, y2 = NODE_POS[nb]
                line = c.create_line(x1, y1, x2, y2,
                                     fill=EDGE_DEF, width=2, tags=('edge', key[0]+'-'+key[1]))
                self._edge_lines[key] = line
                # Distance label
                mx, my = (x1+x2)//2, (y1+y2)//2
                c.create_text(mx, my-8, text=str(dist),
                              fill=TEXT_DIM, font=("Courier", 7))

        # Draw nodes
        for city, (x, y) in NODE_POS.items():
            oval = c.create_oval(x-r, y-r, x+r, y+r,
                                 fill=NODE_DEF, outline=NODE_BORDER, width=2,
                                 tags=('node', city))
            lbl = c.create_text(x, y+r+10, text=city,
                                 fill=TEXT_DIM, font=("Courier", 8),
                                 tags=('label', city))
            self._node_ovals[city] = oval
            self._node_labels[city] = lbl
            self._node_colors[city] = NODE_DEF

    # ── Event Handlers ───────────────────────────────────────────────────────

    def _on_submit(self):
        if self._running:
            return
        src = self.src_var.get()
        dst = self.dst_var.get()
        if src == dst:
            messagebox.showwarning("Same node", "Source and destination must differ.")
            return
        self._on_reset(keep_selection=True)
        self._anim_speed = self.speed_var.get()
        self.submit_btn.config(state='disabled')
        self._running = True
        self._thread = threading.Thread(target=self._run_iddfs,
                                        args=(src, dst), daemon=True)
        self._thread.start()

    def _on_reset(self, keep_selection=False):
        self._running = False
        for city in GRAPH:
            self._set_node_color(city, NODE_DEF, TEXT_DIM)
        for key in self._edge_lines:
            self.canvas.itemconfig(self._edge_lines[key], fill=EDGE_DEF, width=2)
        self.status_var.set("Select source & destination, then press SUBMIT.")
        self.submit_btn.config(state='normal')

    # ── Animation Thread ─────────────────────────────────────────────────────

    def _run_iddfs(self, src, dst):
        gen = iddfs(GRAPH, src, dst)
        try:
            while self._running:
                event = next(gen)
                self._process_event(event, src, dst)
                time.sleep(self._anim_speed / 1000)
        except StopIteration:
            pass
        self._running = False
        self.after(0, lambda: self.submit_btn.config(state='normal'))

    def _process_event(self, event, src, dst):
        kind = event[0]

        if kind == 'new_limit':
            depth = event[1]
            # Reset non-source/dest nodes
            for city in GRAPH:
                if city != src and city != dst:
                    self._set_node_color(city, NODE_DEF, TEXT_DIM)
            for key in self._edge_lines:
                self.canvas.itemconfig(self._edge_lines[key], fill=EDGE_DEF, width=2)
            self._set_node_color(src, COLOR_SOURCE, TEXT_LIGHT)
            self._set_node_color(dst, COLOR_DEST, TEXT_LIGHT)
            self._update_status(f"📏  Depth limit = {depth}  |  Exploring from {src} → {dst}")

        elif kind == 'visit':
            node, path = event[1], event[2]
            if node != src and node != dst:
                self._set_node_color(node, COLOR_VISIT, TEXT_LIGHT)
            # Colour the path edges purple
            for i in range(len(path)-1):
                self._set_edge_color(path[i], path[i+1], COLOR_PATH, 3)
            self._update_status(f"🔍  Visiting: {node}   |   Path: {' → '.join(path)}")

        elif kind == 'backtrack':
            node, path = event[1], event[2]
            if node != src and node != dst:
                self._set_node_color(node, COLOR_DEAD, TEXT_DIM)
            self._update_status(f"↩  Backtracking from: {node}   |   Path: {' → '.join(path)}")

        elif kind == 'found':
            sol_path = event[1]
            # Mark all non-path nodes as dead
            sol_set = set(sol_path)
            for city in GRAPH:
                if city not in sol_set:
                    self._set_node_color(city, COLOR_DEAD, TEXT_DIM)
            # Highlight solution edges and nodes
            for i in range(len(sol_path)-1):
                self._set_edge_color(sol_path[i], sol_path[i+1], COLOR_FINAL, 4)
            for city in sol_path:
                self._set_node_color(city, COLOR_FINAL, TEXT_LIGHT)
            # Source green, dest stays blue as final
            self._set_node_color(sol_path[0], COLOR_SOURCE, TEXT_LIGHT)
            self._update_status(
                f"✅  Solution found!   Path: {' → '.join(sol_path)}   "
                f"(Length: {len(sol_path)-1} hops)"
            )

        elif kind == 'not_found':
            self._update_status(f"❌  No path found from {src} to {dst}.")

    # ── Helpers ──────────────────────────────────────────────────────────────

    def _set_node_color(self, city, fill, text_color=TEXT_LIGHT):
        self.after(0, lambda c=city, f=fill, t=text_color:
                   (self.canvas.itemconfig(self._node_ovals[c], fill=f),
                    self.canvas.itemconfig(self._node_labels[c], fill=t)))

    def _set_edge_color(self, a, b, color, width=2):
        key = tuple(sorted([a, b]))
        if key in self._edge_lines:
            line = self._edge_lines[key]
            self.after(0, lambda l=line, c=color, w=width:
                       self.canvas.itemconfig(l, fill=c, width=w))

    def _update_status(self, msg):
        self.after(0, lambda m=msg: self.status_var.set(m))


# ─── Main ────────────────────────────────────────────────────────────────────

if __name__ == '__main__':
    app = RomaniaIDDFS()
    app.mainloop()